# Building the API layer

The goal here is to make requests to the API, receive a JSON, and save it. A thin wrapper around HTTP requests.

To include the parent folder in the path:

In [6]:
import os
import sys
from pathlib import Path # Path is very nice for handling file paths in a cross-platform way - not to have absolute paths around in the code!

sys.path.append(str(Path().resolve().parent))

In [7]:
RAW_DATA_DIR = Path().resolve().parent / "data" / "raw"

In [8]:
Path().resolve().parent

PosixPath('/home/armankorajac/flixbus_tracker')

In [ ]:
import requests
from config import BASE_URL, STATIONS, API_KEY #import the sensitive API_KEY from the config file, which in turn loads it from the .env file
from datetime import datetime, timedelta, timezone
import time

import json

Reload the config.py if you changed something

In [6]:
import importlib
import config
importlib.reload(config) #reload the config module to ensure we have the latest changes, especially

<module 'config' from '/home/armankorajac/flixbus_tracker/config.py'>

We inserted the station IDs manually of just a couple of stations in Italy. 

Format the time and date. We have to take good care of this, because we will maybe grab data every minute/5 minutes or whatever. They should be dynamic. 
So:
1) Let's call the function to take data every 5 minutes
2) Take data in a window between now - 30 minutes to now + 4 hours or something.
We want to take data on the temporal evolution and stuff later on maybe. For now, maybe, the most important thing is just the final delay information. 

In [7]:
#Flixbus time format is ISO 8601, i.e. this one below, so we need to convert our format to this one
#time_from = "2026-05-07T17:48:00.000Z"
#time_to = "2026-05-07T20:18:00.000Z"


#Function for formatting the time for the flixbus API and for the filenames
def format_time_for_flixbus_api(time):
    timeflix = time.isoformat(timespec="minutes") + ":00.000Z" #get the current time in ISO 8601 format, which is the format required by the FLIXBUS API
    timeflix=timeflix.replace(":", "%3A") #replace the ":" with "%3A" to make it URL safe, as required by the FLIXBUS API
    
    timefile = time.isoformat(timespec="minutes").replace(":", "_").replace("-", "_") #replace the ":" and "-" with "_" to make it a valid filename
    return timeflix, timefile

#print("The correct format \n" + f"2026-05-07T17:48:00.000Z".replace(":", "%3A")) #check if the formatting is correct, it should print "2026-05-07T17%3A48%3A00.000Z"

#print(time_now)
#print(time_now_file)

In [8]:
STATIONS['genoa_principe']

'dcc02ea8-9603-11e6-9066-549f350fcb0c'

In [ ]:
def get_departures(station_name):

    station_id = STATIONS[station_name]

    #Make it dynamic in time when you call the function 
    now = datetime.now()
    print("Current time: ", now)
    _, time_now_file = format_time_for_flixbus_api(now)
    time_in_4_hours, time_in_4_hours_file = format_time_for_flixbus_api(now + timedelta(hours=4))
    time_before_1_hour, time_before_1_hour_file = format_time_for_flixbus_api(now - timedelta(hours=1))


    #The URL looks like this: 
    #url=https://global.api.flixbus.com/gis/v2/timetable/dcbda963-9603-11e6-9066-549f350fcb0c/departures?from=2026-05-07T17%3A48%3A00.000Z&to=2026-05-07T20%3A18%3A00.000Z&apiKey=7781b8fa-07cf-4ab7-8b62-1f3178523ba0
    #the structure is base_url + /gis/v2/timetable/ + station_id + /departures?from= + from_time + &to= + to_time + &apiKey= + API_KEY

    url = f"{BASE_URL}/gis/v2/timetable/{station_id}/departures?from={time_before_1_hour}&to={time_in_4_hours}&apiKey={API_KEY}"
    print(url) #print the URL to check if it's correct

    response = requests.get(url)

    response.raise_for_status() #raise an error if the response is not successful

    json_data = response.json() #get the response in JSON format

    json_data["snapshot_time"] = now.isoformat(timespec="seconds")  #add the snapshot time to the JSON data, in ISO 8601 format



    with open(f"{RAW_DATA_DIR}/{station_name}_departures_{time_before_1_hour_file}_to_{time_in_4_hours_file}_snapshotted_{time_now_file}.json", "w") as f:
        json.dump(json_data, f, indent=4)

    


Call the function and put it in a loop to be called every 10 minutes or so:

In [10]:
RAW_DATA_DIR

PosixPath('/home/armankorajac/flixbus_tracker/data/raw')

In [11]:
STATIONS

{'genoa_principe': 'dcc02ea8-9603-11e6-9066-549f350fcb0c',
 'pisa_pietrasantina': 'dcc1a1c8-9603-11e6-9066-549f350fcb0c',
 'trieste_autostazione': 'dcbda963-9603-11e6-9066-549f350fcb0c',
 'milan_lampugnano': 'dcbc484a-9603-11e6-9066-549f350fcb0c',
 'napoli_centrale': 'dcc19a38-9603-11e6-9066-549f350fcb0c'}

In [12]:
#For now we will not loop through the stations, later on we can
station1 = "genoa_principe"
station2 = 'milan_lampugnano'

while True:

    get_departures(station1)
    get_departures(station2)
    print("Sleeping for 10 minutes...")
    time.sleep(10 * 60) #sleep for 10 minutes before making the next API call



Current time:  2026-05-17 17:33:15.353565
https://global.api.flixbus.com/gis/v2/timetable/dcc02ea8-9603-11e6-9066-549f350fcb0c/departures?from=2026-05-17T16%3A33%3A00.000Z&to=2026-05-17T21%3A33%3A00.000Z&apiKey=7781b8fa-07cf-4ab7-8b62-1f3178523ba0
Current time:  2026-05-17 17:33:16.180094
https://global.api.flixbus.com/gis/v2/timetable/dcbc484a-9603-11e6-9066-549f350fcb0c/departures?from=2026-05-17T16%3A33%3A00.000Z&to=2026-05-17T21%3A33%3A00.000Z&apiKey=7781b8fa-07cf-4ab7-8b62-1f3178523ba0
Sleeping for 10 minutes...
Current time:  2026-05-17 17:43:16.445418
https://global.api.flixbus.com/gis/v2/timetable/dcc02ea8-9603-11e6-9066-549f350fcb0c/departures?from=2026-05-17T16%3A43%3A00.000Z&to=2026-05-17T21%3A43%3A00.000Z&apiKey=7781b8fa-07cf-4ab7-8b62-1f3178523ba0
Current time:  2026-05-17 17:43:16.660344
https://global.api.flixbus.com/gis/v2/timetable/dcbc484a-9603-11e6-9066-549f350fcb0c/departures?from=2026-05-17T16%3A43%3A00.000Z&to=2026-05-17T21%3A43%3A00.000Z&apiKey=7781b8fa-07cf-4ab7

KeyboardInterrupt: 

Let's check how this looks like

In [7]:
departs.keys()


dict_keys(['rides', 'station'])

In [134]:
departs["rides"][5].keys()

dict_keys(['id', 'status', 'platform', 'line', 'location', 'calls', 'vehicle'])

In [129]:
departs["rides"][2]["id"]

'4b113bf2-6f4c-4a58-927f-47747ae4cc14'

In [136]:
departs['rides'][2]['platform']

Okay, we have a mechanism to get the raw .json files.

Now, how are the different JSON files connected? 
Note that Flixbus gives us 2 major datasets:
1) One is v2/timetable, which for every station gives us information about arrivals/departures from this city 
2) The other is v3/ride, which gives us information about RIDES and the stations inbetween that they transverse

From v2/timetable, which has the big array rides = [], we find the unique IDs of the RIDES themselves. 
Once we have these unique IDs, we can go to v3/ride and check anything we want about that specific ride (estimated time of arrival, real time of arrival/departure, whatever there is...).
Already the first file gives a lot of information about delays btw! 

What is to do then? 
I should:
1) Make this API call automatic, like every 5 minutes, and get the raw data
2) Should I then process the data and get info about delays at different timestamps? 


Ah btw, what about the get info on rides function? Because that would be also nice to have? Or actually why...because...once I search for a station (just like the flixbus webpage), I'll show all rides - then you'll find also the rides. It's really just a question how you want to cover the phase-space.